## Step 1: Load Data and Initial Inspection

We load the raw Diabetes 130-US Hospitals dataset and take a first look at its
shape, columns, and data types before doing any cleaning. This tells us what
we're actually working with — row/column counts, which columns are numeric vs.
categorical, and gives a first read on missingness (note: this dataset uses
`'?'` as a placeholder for missing values rather than a true NaN, so `.info()`
won't catch it yet — we'll handle that in the next step).

In [1]:
import pandas as pd

# Load the raw dataset
df = pd.read_csv("../data/diabetic_data.csv")

# Basic shape and structure
print("Shape:", df.shape)
print("\nColumn dtypes:\n", df.dtypes)
print("\nFirst few rows:")
df.head()

Shape: (101766, 50)

Column dtypes:
 encounter_id                int64
patient_nbr                 int64
race                          str
gender                        str
age                           str
weight                        str
admission_type_id           int64
discharge_disposition_id    int64
admission_source_id         int64
time_in_hospital            int64
payer_code                    str
medical_specialty             str
num_lab_procedures          int64
num_procedures              int64
num_medications             int64
number_outpatient           int64
number_emergency            int64
number_inpatient            int64
diag_1                        str
diag_2                        str
diag_3                        str
number_diagnoses            int64
max_glu_serum                 str
A1Cresult                     str
metformin                     str
repaglinide                   str
nateglinide                   str
chlorpropamide                str
glimepiride

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Interpretation: Initial Inspection

The dataset loaded as expected: 101,766 rows and 50 columns, matching the
documented size. A few things stand out:

- **`weight` is already visibly `'?'`-heavy** in just the first 5 rows — this
  confirms what we expected; it's likely to be dropped entirely in the next step.
- **`age` is binned into 10-year ranges** (e.g. `[0-10)`, `[10-20)`) rather than
  given as a raw number — this is already a usable categorical feature, no
  extra binning needed on our end.
- **`readmitted`** (our target) is a 3-class string field (`NO`, `>30`, `<30`)
  — confirms we'll need to binarize it into `readmitted_30d` (1 = `<30`, 0 = 
  otherwise) as planned.
- **40 of the 50 columns are `str` (object) dtype**, mostly the individual
  medication columns (`metformin`, `insulin`, etc.) and diagnosis codes — these
  are almost all categorical, so dtype conversion isn't a concern yet, but we
  do need to check how many distinct values each one takes before deciding
  which are useful.
- No missingness is visible yet via `.dtypes` or `.head()` because `'?'` is a
  valid string, not a null — this is exactly why Step 2 is a dedicated pass to
  replace `'?'` with real `NaN` values and measure missingness properly.

## Step 2: Handle Missing Value Placeholders

This dataset encodes missing values as the string `'?'` instead of a true NaN.
We replace all `'?'` occurrences with `NaN`, then compute the percentage of
missing values per column so we can decide what to drop vs. keep vs. impute.

In [2]:
import numpy as np

# Replace '?' with proper NaN
df = df.replace('?', np.nan)

# Missingness per column, sorted descending
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
print(missing_pct)

weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
dtype: float64


## Interpretation: Missingness Check

Missingness breaks into three tiers:

- **Unusable (drop the column):** `weight` (96.9%) and `max_glu_serum` (94.7%)
  are missing for almost the entire dataset — not enough signal to keep.
- **Heavy but potentially meaningful:** `A1Cresult` (83.3%) and `payer_code`
  (39.6%). For `A1Cresult` specifically, "not measured" is itself informative
  in this dataset — the original study this data comes from (Strack et al.,
  2014) found whether HbA1c was tested at all predicts readmission risk. So
  this one is a candidate to keep as a category ("Not tested") rather than
  drop.
- **Low missingness, easy fixes:** `race` (2.2%), `diag_1/2/3` (≤1.4%) —
  small enough to just drop the affected rows or fill with "Unknown" without
  losing meaningful data.

**The one that actually matters for our model: `medical_specialty` is 49%
missing** — this is our planned grouping variable for the hierarchical model,
so this isn't a minor cleanup decision, it's a modeling decision. Two options:

1. **Keep it, treat missing as its own group** ("Unknown specialty") — 
   preserves all 101,766 rows, and "specialty wasn't recorded" becomes a 
   legitimate 20th+ group in the hierarchy.
2. **Drop the missing rows** — loses roughly half the dataset, which weakens 
   the whole point of a large-N hierarchical model.

**Recommendation: Option 1** — keep all rows, encode missing `medical_specialty`
as `"Unknown"`. It's cleaner than throwing away half your data, and an
"Unknown" group is a legitimate real-world category anyway (specialty often 
isn't recorded consistently across hospitals).

In [3]:
# Drop unusable columns
df = df.drop(columns=['weight', 'max_glu_serum'])

# Keep A1Cresult, fill missing as its own category
df['A1Cresult'] = df['A1Cresult'].fillna('Not tested')

# Grouping variable: fill missing medical_specialty as 'Unknown'
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')

# payer_code: not needed for our research question, drop
df = df.drop(columns=['payer_code'])

# Low-missingness columns: drop the few affected rows
df = df.dropna(subset=['race', 'diag_1', 'diag_2', 'diag_3'])

print("Shape after cleaning:", df.shape)
print("\nRemaining missing values:\n", df.isnull().sum().sum())

Shape after cleaning: (98053, 47)

Remaining missing values:
 0


## Interpretation:

Cleaning worked as planned: 101,766 → 98,053 rows (a ~3.7% loss, entirely
from the small `race`/`diag_1/2/3` drops), 50 → 47 columns (dropped `weight`,
`max_glu_serum`, `payer_code`), and zero missing values remain. This is a
solid, complete dataset to build on.

Two things left before this is truly model-ready: we haven't looked at
**how many distinct specialties** `medical_specialty` actually has (this
directly determines how "hierarchical" our grouping structure really is),
and we haven't touched **`patient_nbr` duplicates** or **binarized the
target** yet.

## Step 4: Inspect Grouping Variable and Target Distribution

Before modeling, we need to know how many specialty groups we're actually
working with (too many tiny groups may need bucketing), and how imbalanced
our target is (readmission classes are rarely balanced, which affects model
choice later).

In [4]:
# How many distinct specialties, and their sizes
specialty_counts = df['medical_specialty'].value_counts()
print("Number of distinct specialties:", df['medical_specialty'].nunique())
print("\nTop 15 specialties by count:\n", specialty_counts.head(15))
print("\nSpecialties with fewer than 50 encounters:", (specialty_counts < 50).sum())

# Raw readmitted distribution
print("\nReadmitted distribution:\n", df['readmitted'].value_counts())
print("\nReadmitted distribution (%):\n", df['readmitted'].value_counts(normalize=True) * 100)

Number of distinct specialties: 73

Top 15 specialties by count:
 medical_specialty
Unknown                            48318
InternalMedicine                   13967
Emergency/Trauma                    7472
Family/GeneralPractice              7140
Cardiology                          5218
Surgery-General                     2969
Nephrology                          1581
Orthopedics                         1355
Orthopedics-Reconstructive          1150
Radiologist                         1115
Pulmonology                          832
Psychiatry                           821
Urology                              636
ObstetricsandGynecology              634
Surgery-Cardiovascular/Thoracic      625
Name: count, dtype: int64

Specialties with fewer than 50 encounters: 41

Readmitted distribution:
 readmitted
NO     52338
>30    34649
<30    11066
Name: count, dtype: int64

Readmitted distribution (%):
 readmitted
NO     53.377255
>30    35.337012
<30    11.285733
Name: proportion, dtype: float64

## Interpretation: Grouping Variable and Target Distribution

This is genuinely a textbook hierarchical modeling setup:

- **73 distinct specialties, wildly uneven in size.** `Unknown` alone accounts
  for 48,318 rows (~49% of the data) — expected, since that's where all our
  filled-in missing values landed. After that, a handful of specialties
  (`InternalMedicine`, `Emergency/Trauma`, `Family/GeneralPractice`,
  `Cardiology`) carry most of the remaining volume, while **41 of the 73
  specialties have fewer than 50 encounters each.**
- **This imbalance is exactly why we're using partial pooling.** A no-pooling
  model would produce wild, unreliable estimates for those 41 small
  specialties (a group with 8 patients could look "high risk" purely by
  chance). A complete-pooling model would ignore specialty-level differences
  entirely. Partial pooling lets small specialties borrow statistical
  strength from the overall population while still letting large groups
  (like `InternalMedicine`, n=13,967) speak mostly for themselves. This
  finding is worth a slide on its own — it's the clearest justification for
  why the topic fits this data.
- **Target is imbalanced:** only 11.3% of encounters are readmitted within
  30 days, 35.3% readmitted after 30 days, and 53.4% not readmitted at all.
  Once we binarize to `<30` vs. everything else, we'll have an ~11%/89%
  split — worth keeping in mind for model evaluation later (accuracy alone
  would be misleading; we'll want to look at posterior predictive checks
  more carefully than a simple accuracy score).

One open decision: with 41 near-empty specialty groups, we may want to set a
minimum-count threshold (e.g. fold anything under 30–50 encounters into an
"Other" bucket) purely to keep the model computationally manageable — but
this is optional since partial pooling is designed to handle small groups
gracefully. Worth deciding once we're actually fitting the model in Stage 4.

In [5]:
# Binarize target: 1 = readmitted within 30 days, 0 = otherwise
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)

print("Binarized target distribution:\n", df['readmitted_30d'].value_counts(normalize=True) * 100)

# Check patient_nbr duplicates
print("\nTotal rows:", len(df))
print("Unique patients:", df['patient_nbr'].nunique())
print("Rows from repeat patients:", len(df) - df['patient_nbr'].nunique())

Binarized target distribution:
 readmitted_30d
0    88.714267
1    11.285733
Name: proportion, dtype: float64

Total rows: 98053
Unique patients: 68630
Rows from repeat patients: 29423


## Interpretation: Binarized Target and Patient Duplicates

The binarized target confirms an 88.7% / 11.3% split — as expected from the
raw distribution. This confirms we'll need to be careful with model
evaluation later (a model that just predicts "no readmission" every time
would already be 88.7% "accurate" while being useless).

**The bigger issue: 68,630 unique patients but 98,053 rows — meaning 29,423
rows (30%) are repeat encounters from patients who show up more than once.**
This matters more than it might look:

- Our model assumes each row is an independent encounter, but repeat
  encounters from the same patient aren't independent — someone who's been
  readmitted before is mechanically more likely to be readmitted again.
  Left as-is, this **inflates the model's confidence** (effective sample
  size is smaller than 98,053 suggests) and can bias specialty-level
  estimates if certain patients cluster in certain specialties.
- This is a classic real decision point, not a bug to silently fix. Two
  reasonable options:
  1. **Keep one encounter per patient** (e.g. first or most recent) —
     clean independence assumption, but throws away 30% of the data and
     loses information about repeat-admission patterns, which are
     arguably central to a readmission study.
  2. **Keep all encounters, note the non-independence as a limitation** —
     more data, more statistical power for the small specialty groups, but
     needs an honest caveat in the interpretation section (and technically
     an argument for patient-level random effects too, which is out of
     scope for this project).

**Recommendation: Option 2** — keep all encounters. For a course project 
with limited weeks, adding a second grouping level (patient nested in
specialty) is real added complexity but likely more than what's needed;
noting the limitation explicitly in your final write-up is honest, defensible, 
and actually strengthens your "limitations and possible extensions" slide —
which is one of the graded rubric lines.

## Step 6: Remove Expired/Hospice Encounters

`discharge_disposition_id` includes codes for patients who died or were
discharged to hospice (codes 11, 19, 20, 21 in the dataset's mapping). These
encounters cannot be readmitted by definition, so keeping them would bias
the target distribution and any specialty-level readmission estimates. We
remove them before finalizing the cleaned dataset.

In [6]:
expired_hospice_codes = [11, 19, 20, 21]

n_before = len(df)
df = df[~df['discharge_disposition_id'].isin(expired_hospice_codes)]
n_after = len(df)

print(f"Rows removed: {n_before - n_after}")
print(f"Shape after removing expired/hospice: {df.shape}")
print("\nUpdated target distribution:\n", df['readmitted_30d'].value_counts(normalize=True) * 100)

Rows removed: 1616
Shape after removing expired/hospice: (96437, 48)

Updated target distribution:
 readmitted_30d
0    88.525151
1    11.474849
Name: proportion, dtype: float64


## Interpretation: Expired/Hospice Removal

1,616 rows removed (about 1.6% of the data) — a small but meaningful cleanup.
The target distribution barely shifted (88.7% → 88.5% / 11.3% → 11.5%),
which makes sense given how few rows were affected, but the fix matters on
principle: every remaining row now represents an encounter where readmission
was actually *possible*, so the model isn't learning from cases that could
only ever be "not readmitted" by definition.

Final shape: **96,437 rows × 48 columns, zero missing values, target
correctly defined, grouping variable justified.** Preprocessing is complete.

## Step 7: Save Cleaned Dataset

Save the preprocessed dataset so downstream notebooks (EDA, modeling) load
this cleaned version directly instead of re-running the full cleaning
pipeline each time.

In [7]:
df.to_csv("../data/diabetic_data_cleaned.csv", index=False)
print("Saved. Final shape:", df.shape)

Saved. Final shape: (96437, 48)


In [8]:
# Check gender anomalies
print("Gender values:\n", df['gender'].value_counts())

# Check for fully duplicate rows
print("\nFully duplicate rows:", df.duplicated().sum())

Gender values:
 gender
Female             52001
Male               44435
Unknown/Invalid        1
Name: count, dtype: int64

Fully duplicate rows: 0


## Interpretation: Final Cleaning Checks

Only 1 row with `Unknown/Invalid` gender — negligible, safe to drop. No
fully duplicate rows exist, confirming there's no leftover data-entry
duplication to worry about.

With this, preprocessing is genuinely comprehensive: missing values handled
and justified column-by-column, grouping variable's missingness resolved
with a defensible decision, target binarized and validated, expired/hospice
encounters removed, patient-duplication issue explicitly documented as a
limitation rather than silently ignored, and now this final gender anomaly
cleared.

In [9]:
# Drop the single Unknown/Invalid gender row
df = df[df['gender'] != 'Unknown/Invalid']

print("Final shape:", df.shape)

# Re-save the cleaned dataset with this last fix included
df.to_csv("../data/diabetic_data_cleaned.csv", index=False)
print("Saved.")

Final shape: (96436, 48)
Saved.
